# 07. Muitos testes

Quando você testa **vários** efeitos ou experimentos, a chance de um falso-positivo aparecer por acaso cresce. A correção de múltiplos testes ajusta os p-valores para controlar esse risco. `ExperimentComparison` faz isso ao comparar experimentos independentes.

## Quatro experimentos independentes

Um tem efeito verdadeiro (`feature_A`); os outros três são nulos. Cada experimento é um CRD estimado com `NeymanCI`, que produz um p-valor.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI


def run_experiment(true_ate, seed, n=120):
    rng = np.random.default_rng(seed)
    y0 = rng.normal(size=n)
    y1 = y0 + true_ate
    df = pd.DataFrame({"x": rng.normal(size=n)})
    design = CRD(p=0.5, seed=seed)
    a = design.randomize(df)
    t = a.data_[a.treatment_col_].to_numpy()
    data = a.data_.copy()
    data["y"] = np.where(t == 1, y1, y0)
    a = CRDAssignment(data=data, treatment_col=a.treatment_col_, design=design, seed=seed)
    return NeymanCI(estimator=DifferenceInMeans(outcome_col="y")).fit(a).estimate()


experiments = {
    "feature_A": run_experiment(0.7, seed=1),   # efeito verdadeiro
    "feature_B": run_experiment(0.0, seed=2),   # nulo
    "feature_C": run_experiment(0.0, seed=3),   # nulo
    "feature_D": run_experiment(0.0, seed=4),   # nulo
}
{name: round(res.p_value, 4) for name, res in experiments.items()}

## Comparar com correção (Holm)

`ExperimentComparison` coleta os p-valores e aplica a correção na família toda. A coluna `p_value_corrected` é o que decide a significância.

In [ ]:
from skxperiments.pipeline import ExperimentComparison

comp = ExperimentComparison(correction="holm", alpha=0.05).run(experiments)
comp.table[["experiment", "ate", "p_value", "p_value_corrected", "significant"]]

## FWER vs. FDR

- **Bonferroni / Holm** controlam o **FWER**: a chance de **qualquer** falso-positivo na família. Mais conservador.
- **Benjamini-Hochberg (BH)** controla o **FDR**: a **proporção** de falsos-positivos entre as descobertas. Mais poder quando há muitos testes.

A escolha depende do custo de um falso-positivo no seu contexto.

In [ ]:
comp_holm = ExperimentComparison(correction="holm").run(experiments)
comp_bh = ExperimentComparison(correction="bh").run(experiments)
print("Significativos (Holm):", comp_holm.significant)
print("Significativos (BH):  ", comp_bh.significant)

## O que aprendemos

- Testar muita coisa infla falsos-positivos; a correção ajusta os p-valores para conter isso.
- FWER (Holm/Bonferroni) é mais estrito; FDR (BH) é mais poderoso quando há muitos testes.
- `ExperimentComparison` aplica a correção e devolve uma tabela pronta para o forest plot.

**Próximo:** `08. Confie no experimento` mostra os diagnósticos (SRM, A/A, balanço).